## Model Development

In [1]:
import torch
import torch.nn as nn

In [2]:
class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )
    
    def forward(self, x):
        return self.conv(x)

In [3]:
class Encoder(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_channels, out_channels)
    
    def forward(self, x):
        x = self.pool(x)
        x = self.conv(x)

        return x


In [4]:
class Decoder(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        
        # Upsample: halves channels, doubles spatial size
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        
        # After concatenating skip connection, in_channels doubles
        self.conv = DoubleConv(out_channels*2, out_channels)
    
    def forward(self, x, skip):
        x = self.up(x)
        x = torch.cat([skip, x], dim=1)
        x = self.conv(x)
        return x

In [5]:
class UNet(nn.Module):
    def __init__(self, in_channels=1, out_channels=1):
        super().__init__()
        
        # Initial double conv (no maxpool)
        self.encoder1 = DoubleConv(in_channels, 64)
        
        # 4 encoder blocks (MaxPool + DoubleConv)
        self.encoder2 = Encoder(64, 128)
        self.encoder3 = Encoder(128, 256)
        self.encoder4 = Encoder(256, 512)
        
        # Bottleneck
        self.bottleneck = Encoder(512, 1024)
        
        # 4 decoder blocks
        self.decoder1 = Decoder(1024, 512)
        self.decoder2 = Decoder(512, 256)
        self.decoder3 = Decoder(256, 128)
        self.decoder4 = Decoder(128, 64)
        
        # Final 1x1 conv
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)
    
    def forward(self, x):
        # Encoder
        skip1 = self.encoder1(x)      
        skip2 = self.encoder2(skip1)  
        skip3 = self.encoder3(skip2)  
        skip4 = self.encoder4(skip3)  
        
        # Bottleneck
        x = self.bottleneck(skip4)
        
        # Decoder - skip connections passed here
        x = self.decoder1(x, skip4)
        x = self.decoder2(x, skip3)
        x = self.decoder3(x, skip2)
        x = self.decoder4(x, skip1)
        
        return self.final_conv(x)

In [6]:
import torch
model = UNet(in_channels=1, out_channels=1)
x = torch.randn(1, 1, 256, 256)  # batch of 1, 1 channel, 256x256
output = model(x)
print("Input shape:", x.shape)
print("Output shape:", output.shape)

Input shape: torch.Size([1, 1, 256, 256])
Output shape: torch.Size([1, 1, 256, 256])


In [9]:
from torchinfo import summary
summary(model, input_size=(1, 1, 256, 256))

Layer (type:depth-idx)                   Output Shape              Param #
UNet                                     [1, 1, 256, 256]          --
├─DoubleConv: 1-1                        [1, 64, 256, 256]         --
│    └─Sequential: 2-1                   [1, 64, 256, 256]         --
│    │    └─Conv2d: 3-1                  [1, 64, 256, 256]         640
│    │    └─BatchNorm2d: 3-2             [1, 64, 256, 256]         128
│    │    └─ReLU: 3-3                    [1, 64, 256, 256]         --
│    │    └─Conv2d: 3-4                  [1, 64, 256, 256]         36,928
│    │    └─BatchNorm2d: 3-5             [1, 64, 256, 256]         128
│    │    └─ReLU: 3-6                    [1, 64, 256, 256]         --
├─Encoder: 1-2                           [1, 128, 128, 128]        --
│    └─MaxPool2d: 2-2                    [1, 64, 128, 128]         --
│    └─DoubleConv: 2-3                   [1, 128, 128, 128]        --
│    │    └─Sequential: 3-7              [1, 128, 128, 128]        221,952
├─E